# 🇮🇳 AI NSE Trading Research System
### Production-grade ML pipeline — RESEARCH USE ONLY

⚠️ **DISCLAIMER**: Does NOT guarantee profits. Significant financial risk involved.


In [ ]:
# STEP 0 — Mount Google Drive
from google.colab import drive
drive.mount("/content/drive")
import os
DRIVE_DIR = "/content/drive/MyDrive/ai-nse-trading"
for d in ["models","logs","data"]: os.makedirs(f"{DRIVE_DIR}/{d}", exist_ok=True)
print("Drive mounted.")

In [ ]:
# STEP 1 — Clone repo + install
REPO_URL = "https://github.com/manoj23798/ai-nse-trading.git"
REPO_DIR = "/content/ai-nse-trading"
import os, sys
if not os.path.exists(REPO_DIR):
    os.system(f"git clone {REPO_URL} {REPO_DIR}")
else:
    os.system(f"cd {REPO_DIR} && git pull")
os.chdir(REPO_DIR)
# Handle nested folder structure
if os.path.exists(os.path.join(REPO_DIR, "ai-nse-trading")):
    REPO_DIR = os.path.join(REPO_DIR, "ai-nse-trading")
os.chdir(REPO_DIR)
os.system("pip install -q -r requirements.txt")
sys.path.insert(0, os.path.join(REPO_DIR, "src"))
print("Ready. CWD:", os.getcwd())

In [ ]:
# STEP 2 — Configuration
TICKERS = ["THANGAMAYL.NS"]  # Predict only Thangamayil Jewellery
USE_INTRADAY = True
INTRADAY_INTERVAL = "15m"   # "5m" or "15m"
INTRADAY_PERIOD = "60d"
DAILY_PERIOD = "5y"
WINDOW = 30
USE_TRANSFORMER = False  # True = more power, slightly slower
CAPITAL = 100_000
TRAIN_CONFIG = {"lr":1e-3,"batch_size":64,"epochs":60,"patience":10,"grad_clip":1.0,"weight_decay":1e-4}
print("Config ready. Tickers:", TICKERS)
print(f"Mode: {'INTRADAY' if USE_INTRADAY else 'DAILY'} | Interval: {INTRADAY_INTERVAL if USE_INTRADAY else '1d'}")
print("Prediction target: next candle using latest closed candle data.")

In [ ]:
# STEP 3 — Download NSE Data
import os, sys
# Ensure src is in path
src_path = os.path.join(REPO_DIR, "src")
if src_path not in sys.path:
    sys.path.insert(0, src_path)
import matplotlib; matplotlib.rcParams.update({"figure.facecolor":"#1a1a2e","axes.facecolor":"#16213e","text.color":"white","axes.labelcolor":"white","xtick.color":"white","ytick.color":"white"})
from data_loader import download_daily, download_intraday, time_split
raw_df = download_intraday(TICKERS, interval=INTRADAY_INTERVAL, period=INTRADAY_PERIOD) if USE_INTRADAY else download_daily(TICKERS, period=DAILY_PERIOD)
print(f"Data shape: {raw_df.shape} | Range: {raw_df.date.min()} -> {raw_df.date.max()}")
raw_df.head()

In [ ]:
# STEP 4 — Feature Engineering
import sys, os
sys.path.insert(0, os.path.join(REPO_DIR, "src"))
from features import build_features, FEATURE_COLS, fit_and_scale, make_sequences
feat_df = build_features(raw_df, intraday=USE_INTRADAY)
print(f"Features: {feat_df.shape} | Cols: {len(FEATURE_COLS)}")
feat_df[FEATURE_COLS].describe().round(3)

In [ ]:
# STEP 5 — Visualise Price + Indicators
import sys, os
sys.path.insert(0, os.path.join(REPO_DIR, "src"))
from evaluate import plot_price_with_indicators
PLOT_TIC = TICKERS[0]
plot_price_with_indicators(feat_df[feat_df.tic==PLOT_TIC].tail(120).copy(), PLOT_TIC, save_path=f"logs/indicators_{PLOT_TIC}.png")

In [ ]:
# STEP 6 — Train Direction Models (all tickers)
import numpy as np, os, shutil, sys
sys.path.insert(0, os.path.join(REPO_DIR, "src"))
from models import build_model, load_model, DEVICE
from train import train_model, save_to_drive
from evaluate import plot_training_history
print(f"Device: {DEVICE}")
trained_models = {}
for TIC in TICKERS:
    print(f"\n=== {TIC} ===")
    tic_df = feat_df[feat_df.tic==TIC].copy()
    if len(tic_df) < WINDOW+50: print(f"Skip {TIC}"); continue
    train_df, val_df, test_df = time_split(tic_df)
    scaler_path = f"models/scaler_{TIC}.pkl"
    X_train, X_val, X_test, scaler = fit_and_scale(train_df, val_df, test_df, FEATURE_COLS, method="standard", save_path=scaler_path)
    y_train = train_df.target_direction.values.astype("float32")
    y_val   = val_df.target_direction.values.astype("float32")
    y_test  = test_df.target_direction.values.astype("float32")
    X_tr_seq,y_tr_seq = make_sequences(X_train,y_train,window=WINDOW)
    X_vl_seq,y_vl_seq = make_sequences(X_val,  y_val,  window=WINDOW)
    X_te_seq,y_te_seq = make_sequences(X_test, y_test, window=WINDOW)
    mtype = "transformer" if USE_TRANSFORMER else "lstm_direction"
    model = build_model(mtype, input_size=len(FEATURE_COLS))
    local_ckpt = f"models/{mtype}_{TIC}.pt"
    drive_ckpt = f"{DRIVE_DIR}/models/{mtype}_{TIC}.pt"
    if os.path.exists(drive_ckpt): shutil.copy2(drive_ckpt,local_ckpt); model=load_model(model,local_ckpt); print("  Loaded from Drive.")
    history = train_model(model,X_tr_seq,y_tr_seq,X_vl_seq,y_vl_seq,task="classification",config=TRAIN_CONFIG,save_path=local_ckpt)
    try: save_to_drive(local_ckpt, DRIVE_DIR+"/models"); save_to_drive(scaler_path, DRIVE_DIR+"/models")
    except Exception as e: print(f"Drive save: {e}")
    trained_models[TIC] = {"model":model,"scaler":scaler,"X_test":X_te_seq,"y_test":y_te_seq,"test_df":test_df,"history":history}
    plot_training_history(history, save_path=f"logs/training_{TIC}.png")
print("All models trained!")

In [ ]:
# STEP 7 — Evaluate on Test Set
import sys, os
sys.path.insert(0, os.path.join(REPO_DIR, "src"))
from evaluate import predict_classification, eval_classification, plot_predicted_vs_actual, log_predictions
for TIC, bundle in trained_models.items():
    print(f"\n--- {TIC} ---")
    probs = predict_classification(bundle["model"], bundle["X_test"])
    eval_classification(bundle["y_test"], probs)
    dates = bundle["test_df"]["date"].values[WINDOW:]
    log_predictions(dates, bundle["y_test"], probs, tic=TIC, label="direction")
    plot_predicted_vs_actual(dates, bundle["y_test"], probs, title=f"{TIC} — Prob vs Actual", save_path=f"logs/pred_vs_actual_{TIC}.png")

In [ ]:
# STEP 8 — Train High/Low Model
import sys, os, shutil, joblib
sys.path.insert(0, os.path.join(REPO_DIR, "src"))
from sklearn.preprocessing import MinMaxScaler, StandardScaler
from data_loader import time_split
from features import FEATURE_COLS, make_sequences
from models import build_model
from train import train_model, save_to_drive

if "feat_df" not in globals() or feat_df is None:
    raise RuntimeError("Step 8 needs engineered features. Please run Step 4 first.")

# Rebuild or create scalers so this step can run even after runtime restarts.
trained_models = trained_models if "trained_models" in globals() and trained_models else {}
os.makedirs("models", exist_ok=True)

for TIC in TICKERS:
    if TIC in trained_models and "scaler" in trained_models[TIC]:
        continue

    local_scaler = f"models/scaler_{TIC}.pkl"
    drive_scaler = f"{DRIVE_DIR}/models/scaler_{TIC}.pkl"

    if not os.path.exists(local_scaler) and os.path.exists(drive_scaler):
        shutil.copy2(drive_scaler, local_scaler)

    if os.path.exists(local_scaler):
        scaler = joblib.load(local_scaler)
    else:
        # First-run fallback: fit scaler directly from Step 4 features.
        tic_df = feat_df[feat_df.tic == TIC].copy()
        if len(tic_df) < WINDOW + 50:
            continue
        train_df, _, _ = time_split(tic_df)
        scaler = StandardScaler()
        scaler.fit(train_df[FEATURE_COLS].values)
        joblib.dump(scaler, local_scaler)
        try:
            save_to_drive(local_scaler, DRIVE_DIR + "/models")
        except Exception:
            pass

    trained_models[TIC] = {"scaler": scaler}

if not trained_models:
    raise RuntimeError("Step 8 could not prepare scalers. Check ticker data and run Step 3/Step 4 again.")

hl_models = {}
for TIC in TICKERS:
    if TIC not in trained_models:
        continue
    print(f"\nHigh/Low model: {TIC}")
    tic_df = feat_df[feat_df.tic == TIC].copy()
    train_df, val_df, test_df = time_split(tic_df)
    sc = trained_models[TIC]["scaler"]
    X_train = sc.transform(train_df[FEATURE_COLS].values)
    X_val = sc.transform(val_df[FEATURE_COLS].values)
    X_test = sc.transform(test_df[FEATURE_COLS].values)

    hl_sc = MinMaxScaler()
    y_tr_hl = hl_sc.fit_transform(train_df[["target_high", "target_low"]].values)
    y_vl_hl = hl_sc.transform(val_df[["target_high", "target_low"]].values)
    y_te_hl = hl_sc.transform(test_df[["target_high", "target_low"]].values)

    X_tr_seq, y_tr_hl2 = make_sequences(X_train, y_tr_hl, window=WINDOW)
    X_vl_seq, y_vl_hl2 = make_sequences(X_val, y_vl_hl, window=WINDOW)
    X_te_seq, y_te_hl2 = make_sequences(X_test, y_te_hl, window=WINDOW)

    hl_model = build_model("high_low", input_size=len(FEATURE_COLS))
    hl_path = f"models/high_low_{TIC}.pt"
    train_model(
        hl_model,
        X_tr_seq,
        y_tr_hl2,
        X_vl_seq,
        y_vl_hl2,
        task="regression_hl",
        config={**TRAIN_CONFIG, "epochs": 40},
        save_path=hl_path,
    )
    try:
        save_to_drive(hl_path, DRIVE_DIR + "/models")
    except Exception:
        pass

    hl_models[TIC] = {"model": hl_model, "scaler": hl_sc}

print("High/Low models done!")

In [ ]:
# STEP 9 — Signal Generation + Risk Management
import sys, os
sys.path.insert(0, os.path.join(REPO_DIR, "src"))
from strategy import generate_signals, RISK_CONFIG
signal_dfs = {}
for TIC, bundle in trained_models.items():
    probs = predict_classification(bundle["model"], bundle["X_test"])
    test_aligned = bundle["test_df"].iloc[WINDOW:].copy().reset_index(drop=True)
    sig_df = generate_signals(test_aligned, probs, config={**RISK_CONFIG,"capital":CAPITAL})
    signal_dfs[TIC] = sig_df
    print(f"\n{TIC} signals:")
    print(sig_df[["date","close","signal","prob_up","stop_loss","take_profit"]].tail(5))

In [ ]:
# STEP 10 — Backtesting
import sys, os
sys.path.insert(0, os.path.join(REPO_DIR, "src"))
from backtest import run_backtest, walk_forward_backtest, plot_equity_curve
backtest_results = {}
for TIC, sig_df in signal_dfs.items():
    print(f"\nBacktest: {TIC}")
    result = run_backtest(sig_df, capital=CAPITAL)
    backtest_results[TIC] = result
    plot_equity_curve(result, title=f"{TIC} Equity Curve", save_path=f"logs/equity_{TIC}.png")

In [ ]:
# STEP 11 — Summary + Save to Drive
import pandas as pd, shutil, sys, os
sys.path.insert(0, os.path.join(REPO_DIR, "src"))
rows = [{**r["metrics"], "ticker":t} for t,r in backtest_results.items()]
summary = pd.DataFrame(rows).set_index("ticker")
print("\n📊 BACKTEST SUMMARY\n", summary.to_string())
summary.to_csv(f"{DRIVE_DIR}/backtest_summary.csv")
try: shutil.copytree("logs", DRIVE_DIR+"/logs", dirs_exist_ok=True); print("Logs → Drive")
except Exception as e: print(e)

In [ ]:
# STEP 12 — Daily Continuous Learning Loop
# Run this cell EVERY DAY to update models with new market data
import sys, os, importlib
sys.path.insert(0, os.path.join(REPO_DIR, "src"))
import continuous_learning
importlib.reload(continuous_learning)

continuous_learning.run_daily_loop(
    tickers=TICKERS,
    model_type="lstm_direction",
    retrain_on_last_n_days=90,
    drive_dir=DRIVE_DIR,
    use_intraday=USE_INTRADAY,
)

In [ ]:
# STEP 13 — Time-Aware Daily Output (Today vs Tomorrow / Intraday)
import os, sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from datetime import datetime, time
from zoneinfo import ZoneInfo

sys.path.insert(0, os.path.join(REPO_DIR, "src"))
from evaluate import predict_classification, predict_regression

if "trained_models" not in globals() or not trained_models:
    raise RuntimeError("Run Step 6 first.")

IST = ZoneInfo("Asia/Kolkata")
now_ist = datetime.now(IST)
market_open = time(9, 15)
market_close = time(15, 30)
print(f"Run time (IST): {now_ist.strftime('%Y-%m-%d %H:%M:%S')}")

for TIC, bundle in trained_models.items():
    latest_df = bundle["test_df"].copy().reset_index(drop=True)
    latest_df["date"] = pd.to_datetime(latest_df["date"]).dt.tz_localize(None)

    if len(latest_df) < 2 or len(bundle["X_test"]) < 2:
        raise RuntimeError("Not enough sequence history. Increase data window and rerun Step 3/4/6.")

    # Intraday mode: show candle-wise predictions with x-axis as time (09:15-15:30)
    if USE_INTRADAY:
        aligned = latest_df.iloc[WINDOW:].copy().reset_index(drop=True)
        probs_all = predict_classification(bundle["model"], bundle["X_test"])

        atr_vals = aligned["atr_14"].values if "atr_14" in aligned.columns else aligned["close"].values * 0.01
        atr_pct = np.clip(atr_vals / np.maximum(aligned["close"].values, 1e-9), 0.002, 0.03)
        pred_next_close = aligned["close"].values * (1.0 + ((probs_all - 0.5) * 2.0 * atr_pct))

        aligned["prob_up"] = probs_all
        aligned["pred_next_close"] = pred_next_close
        aligned["session_date"] = aligned["date"].dt.date

        last_session = aligned["session_date"].max()
        session_df = aligned[aligned["session_date"] == last_session].copy()

        print(f"\n{TIC} — Intraday Summary ({INTRADAY_INTERVAL})")
        print(f"Session date: {last_session}")
        print(f"Candles in session: {len(session_df)}")

        last_row = session_df.iloc[-1]
        next_prob_up = float(last_row["prob_up"])
        next_candle_pred = float(last_row["pred_next_close"])
        print(f"Next Candle Prob(UP): {next_prob_up:.3f}")
        print(f"Next Candle Predicted Close: {next_candle_pred:.2f}")

        # Optional next-candle range from high/low model
        if "hl_models" in globals() and TIC in hl_models:
            hl_bundle = hl_models[TIC]
            hl_scaled = predict_regression(hl_bundle["model"], bundle["X_test"][-1:])
            hl_pred = hl_bundle["scaler"].inverse_transform(hl_scaled.reshape(-1, 2))[0]
            print(f"Next Candle Predicted Range: Low {float(hl_pred[1]):.2f} | High {float(hl_pred[0]):.2f}")

        # Chart: actual close by time + model next-candle prediction by time
        fig, ax = plt.subplots(figsize=(12, 4))
        x = pd.to_datetime(session_df["date"])
        ax.plot(x, session_df["close"].values, color="royalblue", linewidth=1.8, label="Actual Close")
        ax.plot(x, session_df["pred_next_close"].values, color="crimson", linewidth=1.5, linestyle="--", label="Predicted Next Close")

        # Force x-axis display as intraday time
        session_start = pd.Timestamp.combine(pd.Timestamp(last_session), market_open)
        session_end = pd.Timestamp.combine(pd.Timestamp(last_session), market_close)
        ax.set_xlim(session_start, session_end)
        ax.xaxis.set_major_formatter(mdates.DateFormatter("%H:%M"))

        ax.set_title(f"{TIC} — Intraday Prediction ({INTRADAY_INTERVAL})")
        ax.set_xlabel("Time (IST)")
        ax.set_ylabel("Price")
        ax.grid(alpha=0.3)
        ax.legend()
        plt.tight_layout()
        plt.show()
        continue

    # Daily mode (fallback)
    latest_date = latest_df.iloc[-1]["date"].date()
    prev_close = float(latest_df.iloc[-2]["close"])
    latest_close = float(latest_df.iloc[-1]["close"])

    def estimate_close(base_close: float, prob_up: float, atr_value: float) -> float:
        atr_pct_local = float(np.clip(atr_value / max(base_close, 1e-9), 0.005, 0.06))
        move = (prob_up - 0.5) * 2.0 * atr_pct_local
        return base_close * (1.0 + move)

    prob_up_today = float(predict_classification(bundle["model"], bundle["X_test"][-2:-1])[0])
    atr_prev = float(latest_df.iloc[-2]["atr_14"]) if "atr_14" in latest_df.columns else 0.015 * prev_close
    today_pred_close = estimate_close(prev_close, prob_up_today, atr_prev)

    prob_up_tomorrow = float(predict_classification(bundle["model"], bundle["X_test"][-1:])[0])
    atr_latest = float(latest_df.iloc[-1]["atr_14"]) if "atr_14" in latest_df.columns else 0.015 * latest_close
    tomorrow_pred_close = estimate_close(latest_close, prob_up_tomorrow, atr_latest)

    pre_open_mode = now_ist.time() < market_open
    post_close_mode = now_ist.time() >= market_close
    today_close_available = latest_date == now_ist.date()

    print(f"\n{TIC} — Daily Summary")
    if pre_open_mode:
        print("Mode: PRE-MARKET (today prediction only)")
        print(f"Today Predicted Direction Prob(UP): {prob_up_tomorrow:.3f}")
        print(f"Today Predicted Close: {tomorrow_pred_close:.2f}")
    elif post_close_mode and today_close_available:
        print("Mode: AFTER MARKET CLOSE")
        print(f"Today Actual Close ({latest_date}): {latest_close:.2f}")
        print(f"Today Predicted Close (from previous day): {today_pred_close:.2f}")
        print(f"Tomorrow Predicted Direction Prob(UP): {prob_up_tomorrow:.3f}")
        print(f"Tomorrow Predicted Close: {tomorrow_pred_close:.2f}")
    else:
        print("Mode: MARKET HOURS / DATA LAG")
        print(f"Next Session Predicted Direction Prob(UP): {prob_up_tomorrow:.3f}")
        print(f"Next Session Predicted Close: {tomorrow_pred_close:.2f}")